In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import shutil

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Configuration

In [ ]:
# Paths
RAW_DATA_DIR = Path("./data/raw")
PROCESSED_DATA_DIR = Path("./data/processed")
METADATA_FILE = Path("./data/metadata.csv")

# Chili varieties
VARIETIES = ["siling_haba", "siling_labuyo", "super_labuyo"]

# Image settings
IMG_SIZE = (224, 224)
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Split ratios
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")

## 2. Create Sample Data Structure

In [ ]:
def create_data_directories():
    """Create the data directory structure."""
    for variety in VARIETIES:
        for image_type in ["flower", "fruit"]:
            dir_path = RAW_DATA_DIR / variety / image_type
            dir_path.mkdir(parents=True, exist_ok=True)
    
    for split in ["train", "val", "test"]:
        for variety in VARIETIES:
            dir_path = PROCESSED_DATA_DIR / split / variety
            dir_path.mkdir(parents=True, exist_ok=True)
    
    print("Directory structure created!")

create_data_directories()

## 3. Data Collection Summary

In [ ]:
def count_images():
    """Count images in each category."""
    counts = []
    
    for variety in VARIETIES:
        for image_type in ["flower", "fruit"]:
            dir_path = RAW_DATA_DIR / variety / image_type
            if dir_path.exists():
                images = [f for f in dir_path.iterdir() 
                         if f.suffix.lower() in VALID_EXTENSIONS]
                count = len(images)
            else:
                count = 0
            
            counts.append({
                "variety": variety,
                "image_type": image_type,
                "count": count
            })
    
    return pd.DataFrame(counts)

df_counts = count_images()
print("\nImage counts by category:")
print(df_counts.pivot(index='variety', columns='image_type', values='count'))

## 4. Data Augmentation Pipeline

In [ ]:
# Training augmentations
train_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
        A.CLAHE(clip_limit=4.0, p=1),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
    ], p=0.7),
    A.OneOf([
        A.GaussNoise(var_limit=(10.0, 50.0), p=1),
        A.GaussianBlur(blur_limit=3, p=1),
        A.MotionBlur(blur_limit=3, p=1),
    ], p=0.3),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.2),
])

# Validation/Test augmentations (only resize)
val_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
])

print("Augmentation pipelines defined!")

In [ ]:
def augment_image(image_path, transform, n_augmentations=5):
    """Generate augmented versions of an image."""
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    augmented_images = []
    for _ in range(n_augmentations):
        transformed = transform(image=image)
        augmented_images.append(transformed['image'])
    
    return augmented_images

# Demonstrate augmentation
def show_augmentations(image_path, n_samples=6):
    """Visualize augmentations."""
    if not image_path.exists():
        print(f"Image not found: {image_path}")
        return
    
    augmented = augment_image(image_path, train_transform, n_samples)
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, img in zip(axes.flat, augmented):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle('Data Augmentation Examples')
    plt.tight_layout()
    plt.show()

# Uncomment to test with an actual image
# show_augmentations(RAW_DATA_DIR / "siling_labuyo" / "flower" / "sample.jpg")

## 5. Process and Split Data

In [ ]:
def collect_all_images():
    """Collect all image paths with their labels."""
    data = []
    
    for variety in VARIETIES:
        for image_type in ["flower", "fruit"]:
            dir_path = RAW_DATA_DIR / variety / image_type
            if dir_path.exists():
                for img_path in dir_path.iterdir():
                    if img_path.suffix.lower() in VALID_EXTENSIONS:
                        data.append({
                            "path": str(img_path),
                            "variety": variety,
                            "image_type": image_type,
                            "filename": img_path.name
                        })
    
    return pd.DataFrame(data)

df_images = collect_all_images()
print(f"Total images collected: {len(df_images)}")
print("\nDistribution by variety:")
print(df_images['variety'].value_counts())

In [ ]:
def split_and_process_data(df, augment_train=True, n_augmentations=3):
    """Split data into train/val/test and process images."""
    
    if len(df) == 0:
        print("No images to process. Please add images to the data/raw directory.")
        return
    
    # Stratified split
    train_df, temp_df = train_test_split(
        df, test_size=(VAL_RATIO + TEST_RATIO), 
        stratify=df['variety'], random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=TEST_RATIO/(VAL_RATIO + TEST_RATIO),
        stratify=temp_df['variety'], random_state=42
    )
    
    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    
    # Process each split
    for split_name, split_df, transform in [
        ("train", train_df, train_transform),
        ("val", val_df, val_transform),
        ("test", test_df, val_transform)
    ]:
        print(f"\nProcessing {split_name} split...")
        
        for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
            variety = row['variety']
            src_path = Path(row['path'])
            
            # Process original image
            image = cv2.imread(str(src_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            processed = transform(image=image)['image']
            
            # Save original
            dst_path = PROCESSED_DATA_DIR / split_name / variety / row['filename']
            cv2.imwrite(str(dst_path), cv2.cvtColor(processed, cv2.COLOR_RGB2BGR))
            
            # Augment training data
            if split_name == "train" and augment_train:
                for i in range(n_augmentations):
                    aug_image = train_transform(image=image)['image']
                    aug_name = f"{src_path.stem}_aug{i}{src_path.suffix}"
                    aug_path = PROCESSED_DATA_DIR / split_name / variety / aug_name
                    cv2.imwrite(str(aug_path), cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR))
    
    print("\nData processing complete!")

# Uncomment to run processing
# split_and_process_data(df_images)

## 6. Create Metadata CSV

In [ ]:
def create_metadata():
    """Create metadata CSV with image paths and labels."""
    metadata = []
    
    for split in ["train", "val", "test"]:
        for variety in VARIETIES:
            dir_path = PROCESSED_DATA_DIR / split / variety
            if dir_path.exists():
                for img_path in dir_path.iterdir():
                    if img_path.suffix.lower() in VALID_EXTENSIONS:
                        metadata.append({
                            "path": str(img_path),
                            "split": split,
                            "variety": variety,
                            "label": VARIETIES.index(variety),
                            "filename": img_path.name
                        })
    
    df_metadata = pd.DataFrame(metadata)
    df_metadata.to_csv(METADATA_FILE, index=False)
    
    print(f"Metadata saved to {METADATA_FILE}")
    print(f"\nTotal images: {len(df_metadata)}")
    print("\nDistribution:")
    print(df_metadata.groupby(['split', 'variety']).size().unstack(fill_value=0))
    
    return df_metadata

# Uncomment to create metadata
# df_metadata = create_metadata()

## 7. Feature Extraction for Traditional ML

In [ ]:
def extract_color_features(image):
    """Extract color histogram features."""
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    # Calculate histograms
    h_hist = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [16], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [16], [0, 256]).flatten()
    
    # Normalize
    h_hist = h_hist / h_hist.sum()
    s_hist = s_hist / s_hist.sum()
    v_hist = v_hist / v_hist.sum()
    
    return np.concatenate([h_hist, s_hist, v_hist])

def extract_shape_features(image):
    """Extract shape features using contours."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours) == 0:
        return np.zeros(7)
    
    largest = max(contours, key=cv2.contourArea)
    moments = cv2.moments(largest)
    hu_moments = cv2.HuMoments(moments).flatten()
    
    # Log transform for scale invariance
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)
    
    return hu_moments

def extract_all_features(image):
    """Extract all features from an image."""
    color_features = extract_color_features(image)
    shape_features = extract_shape_features(image)
    
    return np.concatenate([color_features, shape_features])

print("Feature extraction functions defined!")
print(f"Total features per image: {48 + 7} = 55")

In [ ]:
def create_feature_dataset(metadata_df):
    """Create feature dataset for traditional ML models."""
    features_list = []
    
    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        image = cv2.imread(row['path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        features = extract_all_features(image)
        features_list.append({
            'path': row['path'],
            'split': row['split'],
            'variety': row['variety'],
            'label': row['label'],
            **{f'feature_{i}': v for i, v in enumerate(features)}
        })
    
    df_features = pd.DataFrame(features_list)
    df_features.to_csv('./data/features.csv', index=False)
    
    print(f"Feature dataset saved with {len(df_features)} samples")
    return df_features

# Uncomment to create features
# df_features = create_feature_dataset(df_metadata)

## 8. Summary

In [ ]:
print("="*50)
print("DATA PREPROCESSING COMPLETE")
print("="*50)
print("\nNext Steps:")
print("1. Add chili images to data/raw/{variety}/{flower|fruit}/")
print("2. Run the split_and_process_data() function")
print("3. Create metadata with create_metadata()")
print("4. Proceed to variety classifier training notebook")